# Spot The Mask Challenge
**Objective**: Predict the likelihood that an image contains at least one person wearing a mask.
**Metric**: Area Under the Curve (AUC)

The dataset contains ~1,300 training images and 509 test images.

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Path Configuration and Data Extraction

In [ ]:
def setup_paths():
    train_csv = 'train_labels.csv'
    images_zip = 'images.zip'
    sample_sub = 'SampleSubmission.csv'
    
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'train_labels.csv' in files:
                train_csv = os.path.join(root, 'train_labels.csv')
            if 'images.zip' in files:
                images_zip = os.path.join(root, 'images.zip')
            if 'SampleSubmission.csv' in files:
                sample_sub = os.path.join(root, 'SampleSubmission.csv')
                
    return train_csv, images_zip, sample_sub

TRAIN_CSV, IMAGES_ZIP, SAMPLE_SUB = setup_paths()
EXTRACT_DIR = '/kaggle/working/images'

def extract_images(zip_path, extract_path):
    if not os.path.exists(extract_path):
        os.makedirs(extract_path, exist_ok=True)
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("Extraction complete.")
    else:
        print("Images already extracted.")

if os.path.exists(IMAGES_ZIP):
    extract_images(IMAGES_ZIP, EXTRACT_DIR)
else:
    print("images.zip not found.")

## 2. Dataset Class

In [ ]:
class MaskDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test
        
        # Build path map once
        self.path_map = {}
        for root, _, files in os.walk(img_dir):
            for f in files:
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    self.path_map[f] = os.path.join(root, f)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['id'] if 'id' in self.df.columns else self.df.iloc[idx]['image_id']
        if not str(img_name).lower().endswith(('.jpg', '.png', '.jpeg')):
            img_name = str(img_name) + '.jpg'
            
        img_path = self.path_map.get(img_name) or os.path.join(self.img_dir, img_name)
        
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        
        if self.transform:
            image = self.transform(image)
            
        if self.is_test:
            return image, self.df.iloc[idx].get('id') or self.df.iloc[idx].get('image_id')
            
        label = self.df.iloc[idx]['target']
        return image, torch.tensor(label, dtype=torch.float32)

## 3. Model Definition (EfficientNet-B0)

In [ ]:
def get_model():
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Linear(num_ftrs, 1),
        nn.Sigmoid()
    )
    return model.to(device)

## 4. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, epochs=5):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
    best_auc = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        model.eval()
        val_preds = []
        val_labels = []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                outputs = model(images)
                val_preds.extend(outputs.cpu().numpy())
                val_labels.extend(labels.numpy())
        
        auc = roc_auc_score(val_labels, val_preds)
        print(f"Epoch {epoch+1}, Train Loss: {train_loss/len(train_loader):.4f}, Val AUC: {auc:.4f}")
        
        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), 'best_mask_model.pth')
    
    return best_auc

## 5. Main Execution and Submission

In [ ]:
if os.path.exists(TRAIN_CSV):
    df = pd.read_csv(TRAIN_CSV)
    train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
    
    transform = T.Compose([
        T.Resize((224, 224)),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    val_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    train_ds = MaskDataset(train_df, EXTRACT_DIR, transform)
    val_ds = MaskDataset(val_df, EXTRACT_DIR, val_transform)
    
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
    
    model = get_model()
    train_model(model, train_loader, val_loader)
    
    if os.path.exists(SAMPLE_SUB):
        test_df = pd.read_csv(SAMPLE_SUB)
        test_ds = MaskDataset(test_df, EXTRACT_DIR, val_transform, is_test=True)
        test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
        
        model.load_state_dict(torch.load('best_mask_model.pth'))
        model.eval()
        
        results = []
        with torch.no_grad():
            for images, ids in tqdm(test_loader, desc="Predicting"):
                images = images.to(device)
                outputs = model(images)
                probs = outputs.cpu().numpy().flatten()
                for img_id, prob in zip(ids, probs):
                    results.append({'id': img_id, 'label': prob})
        
        sub_df = pd.DataFrame(results)
        sub_df.to_csv('submission.csv', index=False)
        print("Submission file generated: submission.csv")
        print(sub_df.head())
else:
    print("train_labels.csv not found.")